# 06. The VXX/SVXY backtest: timing, costs, segments, PSR and DSR

Milestone 6 turns a frozen h = 21 forecast into a position and asks whether it earns anything after
costs, and whether that could be luck:

```
signal at the close of t      d_t = forecast_t - VIX_t (vol points); +1 if d_t > band, -1 if d_t < -band, else 0
fill at the close of t+1      rule L7: never at the close the signal was computed from
earn t+1 -> t+2               +1: 100% VXX (long vol), -1: 100% SVXY (short vol, -0.5x), 0: cash at 0
cost                          (half-spread + commission) per side on |change in weight|
luck                          PSR against 0; DSR against E[max SR] of every trial run on dev
```

The full step (after the forecasts and M5's dev report; local only for real data):

```
uv run python -m ivf.data fetch --symbols VXX SVXY
uv run python -m ivf.backtest prepare
uv run python -m ivf.backtest run [--print-tables]
```

This notebook runs on the **synthetic** market and a toy VXX/SVXY derived from the synthetic VIX
(`ivf.backtest.data.synthetic_tradables`). The forecast is the trailing 21-session realised volatility
(the `rw` model of the rv21 track). **The holdout is not read.**

*This file is generated by `tools/build_notebooks.py`. Edit the spec there.*

In [ ]:
import numpy as np
import pandas as pd

from ivf import protocol
from ivf.backtest import data as bdata
from ivf.backtest import engine, metrics, segments, strategy
from ivf.data import calendar, synthetic

PROTO = protocol.load_protocol(protocol.protocol_path(synthetic=True))
CFG = strategy.load_config()
SEGS = segments.load_segments()
IV, PRICE = synthetic.synthetic_market(PROTO)
HS = PROTO.holdout_start
SESSIONS = [d for d in calendar.master_calendar(PROTO)["date"] if d < HS]  # dev only: nothing on or after holdout_start
IV, PRICE = IV[IV.date < HS], PRICE[PRICE.date < HS]
ETP = bdata.synthetic_tradables(IV)
print("data: SYNTHETIC | sessions", len(SESSIONS), "| last", SESSIONS[-1], "< holdout_start", HS, "| lag", CFG.lag, "| cost", CFG.cost_bps, "bp per side")

## 1. Timing and costs by hand

Seven sessions, a signal at each close. The row dated d holds the signal of origin d - 2. With 10 bp per
side: an entry from cash is one side, a switch VXX -> SVXY two, and the day's net return is
`(1 - cost)(1 + gross) - 1`.

In [ ]:
S = SESSIONS[:7]
VXX = [100.0, 100.0, 110.0, 99.0, 99.0, 108.9, 108.9]
SVXY = [50.0, 50.0, 45.0, 49.5, 49.5, 44.55, 49.005]
prices = pd.DataFrame([{"ticker": t, "date": d, "adj_close": p} for t, path in (("VXX", VXX), ("SVXY", SVXY)) for d, p in zip(S, path)])
rets = pd.DataFrame({t: segments.returns_on_sessions(prices, t, S) for t in ("VXX", "SVXY")})
signal = pd.Series([1.0, 1.0, -1.0, 0.0, -1.0, 0.0, 0.0], index=S)
book = engine.daily(engine.positions(signal, S, S[2:], lag=2), rets, cost_bps=10.0)
print(book[["date", "origin_date", "signal", "gross_ret", "turnover", "cost", "net_ret", "equity"]].round(6).to_string(index=False))
same_day = engine.daily(engine.positions(signal, S, S[2:], lag=0), rets, cost_bps=10.0)
print("same-day fills would give", same_day.net_ret.round(4).tolist(), "instead of", book.net_ret.round(4).tolist())

## 2. Segments: where a return may be used

SVXY's target changed from -1x to -0.5x at the close of 2018-02-27; VXX's usable segment starts at
2018-08-01, its first close after the last stale print on the real data (review A). A return belongs to
a segment when both its closes do, so the first usable return of both is dated 2018-08-02. The checks
look for missing closes, stale prints, unadjusted splits and an exposure that does not match.

In [ ]:
usable = segments.usable_return_dates(SEGS, SESSIONS)
print("usable returns", usable[0], "..", usable[-1], "|", len(usable), "sessions")
violations, rows = segments.check_segments(ETP, SEGS, SESSIONS, SESSIONS[-1], CFG.splits, CFG.tolerance)
print("violations:", violations)
print(pd.DataFrame(rows)[["check", "ticker", "valid_from", "usable", "sessions", "missing", "slope", "expected_slope"]].round(3).to_string(index=False))

## 3. A dev backtest: the band rule against the benchmarks

The forecast is trailing rv21 at each origin; `d = rv21 - VIX`. Four band values are four trials.

In [ ]:
r = np.log(PRICE.set_index("date")["close"]).diff()
rv21 = (100 * np.sqrt(252 / 21 * (r**2).rolling(21).sum())).reindex(SESSIONS)
vix = IV[IV.symbol == "VIX"].set_index("date")["close"]
sig = {f"band_{b:g}": strategy.band_signal(rv21.dropna(), vix, b) for b in (0.0, 1.0, 2.0, 4.0)}
sig.update({"bench_long_vxx": pd.Series(1.0, index=SESSIONS), "bench_long_svxy": pd.Series(-1.0, index=SESSIONS)})
DAILY = strategy.run_period(CFG, SEGS, SESSIONS, ETP, sig, "dev", HS, PROTO.data_end)
rows = [{"strategy_id": k, "cost_bps": c, **metrics.summary(g)} for (k, c), g in DAILY.groupby(["strategy_id", "cost_bps"])]
SUMMARY = pd.DataFrame(rows)
cols = ["strategy_id", "cost_bps", "n_days", "ann_return", "ann_vol", "sharpe_ann", "sortino_ann", "max_drawdown", "n_trades", "hit_rate", "psr0"]
print(SUMMARY[SUMMARY.cost_bps == CFG.cost_bps][cols].round(3).to_string(index=False))

## 4. Deflating for the trials

DSR is PSR against E[max SR] of N trials under the null "every trial has SR 0", with V[SR] the larger of
the trials' spread and 1/T (`docs/concepts/15-psr-dsr-and-trial-counting.md`). Here N = 4 band values.

In [ ]:
at = SUMMARY[SUMMARY.cost_bps == CFG.cost_bps].set_index("strategy_id")
trials = at.loc[[k for k in at.index if k.startswith("band_")]]
for k, row in trials.iterrows():
    d = metrics.deflate(row["sr_daily"], int(row["n_days"]), row["skew"], row["kurt_pearson"], list(trials.sr_daily), len(trials))
    print(f"{k:8} sharpe_ann {row['sharpe_ann']:6.3f}  psr0 {row['psr0']:.3f}  E[max SR] (daily) {d['e_max_sr']:.4f}  dsr {d['dsr']:.3f}")
print("holdout_looks", PROTO.document["holdout_looks"], "| last return read", DAILY.date.max(), "< holdout_start", HS)